In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import LabelEncoder

ROOT         = Path(r"D:\hocki7\Data Annalyst\Detai")
PATH_SAMPLED = ROOT / "1_data" / "processed" / "02_sampled.csv"
PATH_CLEANED = ROOT / "1_data" / "processed" / "03_cleaned.csv"

print("✅ Import xong")

✅ Import xong


In [ ]:
# ================================================================
# ĐỌC THẲNG TỪ FILE 
# ================================================================
df = pd.read_csv(PATH_SAMPLED)
print(f"Đọc xong: {df.shape[0]:,} dòng | {df.shape[1]} cột")

def is_str_col(series):
    return str(series.dtype).lower() in ['object', 'string', 'str']

# ── 1. Bỏ TransactionID ─────────────────────────────────────────
df = df.drop(columns=['TransactionID'], errors='ignore')
print(f"✅ 1. Bỏ TransactionID → còn {df.shape[1]} cột")

# ── 2. Tách TransactionDT ───────────────────────────────────────
if 'TransactionDT' in df.columns:
    df['hour_of_day'] = (df['TransactionDT'] // 3600) % 24
    df['day_of_week'] = (df['TransactionDT'] // 86400) % 7
    df['is_night']    = (
        (df['hour_of_day'] >= 22) | (df['hour_of_day'] <= 5)
    ).astype(int)
    df = df.drop(columns=['TransactionDT'])
print(f"✅ 2. Tách TransactionDT → hour_of_day, day_of_week, is_night")

# ── 3. Log transform TransactionAmt ────────────────────────────
if 'TransactionAmt' in df.columns:
    df['TransactionAmt_log'] = np.log1p(df['TransactionAmt'])
    df = df.drop(columns=['TransactionAmt'])
print(f"✅ 3. Log transform TransactionAmt_log")

# ── 4. Group email domain ───────────────────────────────────────
def group_email(val):
    d = str(val).lower()
    if d in ['nan','unknown','none']: return 'unknown'
    for name in ['gmail','yahoo','hotmail','outlook','icloud','live','msn','anonymous']:
        if name in d: return name
    return 'other'

def group_device(val):
    d = str(val).lower()
    if d in ['nan','unknown','none']: return 'unknown'
    if 'windows' in d:               return 'windows'
    if any(x in d for x in ['ios','iphone','ipad']):           return 'ios'
    if any(x in d for x in ['android','samsung','sm-','huawei']): return 'android'
    if 'mac' in d:  return 'mac'
    if 'linux' in d: return 'linux'
    return 'other'

def group_browser(val):
    d = str(val).lower()
    if d in ['nan','unknown','none']: return 'unknown'
    for name in ['chrome','firefox','safari','edge','ie']:
        if name in d: return name
    return 'other'

for col, fn in [('P_emaildomain', group_email),
                ('R_emaildomain', group_email),
                ('DeviceInfo',    group_device),
                ('id_31',         group_browser)]:
    if col in df.columns:
        df[col] = df[col].fillna('unknown').apply(fn)
print(f"✅ 4. Group email/device/browser")

# ── 5. Xử lý missing ───────────────────────────────────────────
print(f"\n✅ 5. Xử lý missing:")
for col in df.columns:
    if col == 'isFraud':
        continue
    n = df[col].isna().sum()
    if n == 0:
        continue
    if is_str_col(df[col]):
        df[col] = df[col].fillna('Unknown')
        print(f"   {col:<22} str     → 'Unknown'       [{n:,}]")
    else:
        med = float(df[col].median(skipna=True))
        df[col] = df[col].fillna(med)
        print(f"   {col:<22} numeric → median={med:.1f}  [{n:,}]")

# ── 6. Label Encode ─────────────────────────────────────────────
print(f"\n✅ 6. Label Encode:")
le = LabelEncoder()
for col in [c for c in df.columns if is_str_col(df[c])]:
    n_unique         = df[col].nunique()
    df[col + '_enc'] = le.fit_transform(df[col].fillna('Unknown').astype(str))
    df               = df.drop(columns=[col])
    print(f"   {col:<22} → {col}_enc  ({n_unique} unique)")

# ── 7. Đặc trưng tương tác ─────────────────────────────────────
if all(c in df.columns for c in ['C1','C2','C6']):
    df['C_total']   = df['C1'] + df['C2'] + df['C6']
if all(c in df.columns for c in ['D1','D4','D10']):
    df['D_total']   = df['D1'] + df['D4'] + df['D10']
if all(c in df.columns for c in ['dist1','dist2']):
    df['dist_diff'] = (df['dist1'] - df['dist2']).abs()
print(f"\n✅ 7. Thêm C_total, D_total, dist_diff")

# ── KIỂM TRA CUỐI ──────────────────────────────────────────────
print("\n" + "=" * 62)
print("  KẾT QUẢ CUỐI")
print("=" * 62)

total_miss = df.isnull().sum().sum()
str_remain = [c for c in df.columns if is_str_col(df[c])]

print(f"\n  1. Missing = 0     : {total_miss}  {'✅' if total_miss==0 else '❌'}")
print(f"  2. Cột str = 0     : {len(str_remain)}  {'✅' if len(str_remain)==0 else '❌ '+str(str_remain)}")
print(f"  3. Số cột          : {df.shape[1]}  {'✅' if 23<=df.shape[1]<=28 else '⚠️'}")
print(f"  4. Số dòng         : {df.shape[0]:,}")

print(f"\n{'#':<4}{'Tên cột':<28}{'Dtype':>10}{'Min':>10}{'Max':>10}{'Mean':>10}")
print("-" * 72)
for i, col in enumerate(df.columns, 1):
    if pd.api.types.is_numeric_dtype(df[col]):
        print(f"  {i:<3}{col:<28}{str(df[col].dtype):>10}"
              f"{df[col].min():>10.2f}{df[col].max():>10.2f}{df[col].mean():>10.2f}")
    else:
        print(f"  {i:<3}{col:<28}{str(df[col].dtype):>10}"
              f"{'—':>10}{'—':>10}{'—':>10}")

vc = df['isFraud'].value_counts()
print(f"\n  Phân phối nhãn:")
print(f"    Fraud (1)       : {vc.get(1,0):,}  ({vc.get(1,0)/len(df)*100:.1f}%)")
print(f"    Không fraud (0) : {vc.get(0,0):,}  ({vc.get(0,0)/len(df)*100:.1f}%)")

# ── LƯU ────────────────────────────────────────────────────────
df.to_csv(PATH_CLEANED, index=False)
print(f"\n✅ Đã lưu: {PATH_CLEANED}")
print(f"   Kích thước: {PATH_CLEANED.stat().st_size/1024**2:.1f} MB")

Đọc xong: 200,000 dòng | 28 cột
✅ 1. Bỏ TransactionID → còn 27 cột
✅ 2. Tách TransactionDT → hour_of_day, day_of_week, is_night
✅ 3. Log transform TransactionAmt_log
✅ 4. Group email/device/browser

✅ 5. Xử lý missing:
   card4                  str     → 'Unknown'       [518]
   card6                  str     → 'Unknown'       [516]
   addr1                  numeric → median=299.0  [26,136]
   addr2                  numeric → median=87.0  [26,136]
   dist1                  numeric → median=8.0  [122,014]
   dist2                  numeric → median=41.0  [185,528]
   D1                     numeric → median=2.0  [467]
   D4                     numeric → median=21.0  [57,250]
   D10                    numeric → median=10.0  [26,701]
   M4                     str     → 'Unknown'       [91,929]
   M6                     str     → 'Unknown'       [61,804]
   id_12                  str     → 'Unknown'       [146,617]
   id_15                  str     → 'Unknown'       [147,685]
   id_28       

In [3]:
# CELL XÁC NHẬN CUỐI
print(f"1. Missing = 0  : {df.isnull().sum().sum()}  {'✅' if df.isnull().sum().sum()==0 else '❌'}")
print(f"2. Cột str = 0  : {len([c for c in df.columns if str(df[c].dtype) in ['object','string','str']])}  {'✅' if len([c for c in df.columns if str(df[c].dtype) in ['object','string','str']])==0 else '❌'}")
print(f"3. Số cột       : {df.shape[1]}")
print(f"\nDanh sách {df.shape[1]} cột cuối:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2}. {col:<28} {str(df[col].dtype)}")

1. Missing = 0  : 0  ✅
2. Cột str = 0  : 0  ✅
3. Số cột       : 32

Danh sách 32 cột cuối:
   1. isFraud                      int64
   2. card1                        int64
   3. addr1                        float64
   4. addr2                        float64
   5. dist1                        float64
   6. dist2                        float64
   7. C1                           float64
   8. C2                           float64
   9. C6                           float64
  10. D1                           float64
  11. D4                           float64
  12. D10                          float64
  13. hour_of_day                  int64
  14. day_of_week                  int64
  15. is_night                     int64
  16. TransactionAmt_log           float64
  17. ProductCD_enc                int64
  18. card4_enc                    int64
  19. card6_enc                    int64
  20. P_emaildomain_enc            int64
  21. R_emaildomain_enc            int64
  22. M4_enc              

In [4]:
# CELL GỌN CỘT — giữ 25 cột có giá trị nhất
COLS_KEEP = [
    # Nhãn
    'isFraud',

    # Giao dịch
    'TransactionAmt_log',   # số tiền (đã log transform)
    'ProductCD_enc',        # loại sản phẩm

    # Thẻ
    'card1',                # mã thẻ
    'card4_enc',            # thương hiệu thẻ
    'card6_enc',            # loại thẻ debit/credit

    # Địa chỉ
    'addr1',                # vùng người gửi
    'addr2',                # quốc gia

    # Khoảng cách
    'dist1',                # khoảng cách giao dịch
    'dist2',                # khoảng cách thứ 2

    # Email
    'P_emaildomain_enc',    # email người gửi
    'R_emaildomain_enc',    # email người nhận

    # Hành vi (counting)
    'C1', 'C2', 'C6',
    'C_total',              # tổng hành vi

    # Thời gian giao dịch
    'D1', 'D4', 'D10',
    'D_total',              # tổng timedelta

    # Thời gian
    'hour_of_day',
    'day_of_week',
    'is_night',

    # Match features
    'M4_enc',
    'M6_enc',
]

df_final = df[COLS_KEEP].copy()

print(f"✅ Giữ lại {df_final.shape[1]} cột từ {df.shape[1]} cột")
print(f"   Bỏ {df.shape[1] - df_final.shape[1]} cột ít giá trị hơn")
print(f"\nBỏ các cột: {[c for c in df.columns if c not in COLS_KEEP]}")

# Xác nhận
print(f"\n{'='*55}")
print(f"  DATASET CUỐI CÙNG")
print(f"{'='*55}")
print(f"  Số dòng  : {df_final.shape[0]:,}")
print(f"  Số cột   : {df_final.shape[1]}")
print(f"  Missing  : {df_final.isnull().sum().sum()}")
print(f"\n  {'#':<4}{'Tên cột':<25}{'Dtype':>10}")
print(f"  {'-'*42}")
for i, col in enumerate(df_final.columns, 1):
    print(f"  {i:<4}{col:<25}{str(df_final[col].dtype):>10}")

# Lưu đè lên file cleaned
df_final.to_csv(PATH_CLEANED, index=False)
print(f"\n✅ Đã lưu: {PATH_CLEANED}")
print(f"   Kích thước: {PATH_CLEANED.stat().st_size/1024**2:.1f} MB")
print(f"\n🎉 DATASET SẠCH SẴN SÀNG → bắt đầu 02_EDA.ipynb")

✅ Giữ lại 25 cột từ 32 cột
   Bỏ 7 cột ít giá trị hơn

Bỏ các cột: ['id_12_enc', 'id_15_enc', 'id_28_enc', 'id_31_enc', 'DeviceType_enc', 'DeviceInfo_enc', 'dist_diff']

  DATASET CUỐI CÙNG
  Số dòng  : 200,000
  Số cột   : 25
  Missing  : 0

  #   Tên cột                       Dtype
  ------------------------------------------
  1   isFraud                       int64
  2   TransactionAmt_log          float64
  3   ProductCD_enc                 int64
  4   card1                         int64
  5   card4_enc                     int64
  6   card6_enc                     int64
  7   addr1                       float64
  8   addr2                       float64
  9   dist1                       float64
  10  dist2                       float64
  11  P_emaildomain_enc             int64
  12  R_emaildomain_enc             int64
  13  C1                          float64
  14  C2                          float64
  15  C6                          float64
  16  C_total                     float6